# StudentEnv — How to Use

This notebook lets you run a simulated student and generate synthetic diary entries from the simulation.

`StudentEnv` models a student's psychological state over a 30-day period. **You don't need to understand the internals** — treat it as a black box that responds to actions and returns observations.

---

## Observations

Each step returns a 5-value array. All values are between 0 and 1.

| Index | Name | Meaning |
|-------|------|---------|
| 0 | `energy` | How much cognitive capacity the student has left |
| 1 | `stress` | Current stress level (rises passively near deadlines) |
| 2 | `self_efficacy` | Confidence in own ability |
| 3 | `social_need` | Unmet need for social contact (rises passively each day) |
| 4 | `days_until_deadline` | Normalized days remaining until the next deadline |

---

## Actions

Pass an integer 0–5 to `env.step(action)`.

| Action | Name | Effect |
|--------|------|--------|
| 0 | `study` | Costs energy; reduces stress and builds confidence — unless already overwhelmed |
| 1 | `attend_lecture` | Moderate energy cost; small stress and confidence boost |
| 2 | `leisure` | Restores energy and reduces stress; slight confidence dip |
| 3 | `socialize` | Reduces social need; small confidence gain |
| 4 | `rest` | Strong energy recovery; moderate stress relief |
| 5 | `skip` | Short-term energy gain; increases stress, reduces confidence |

---

## Reward

`reward = self_efficacy - stress`

A positive reward means the student is confident and not overwhelmed. A negative reward means the reverse.

In [ ]:
# Setup — run once, no need to modify
import gymnasium as gym
import numpy as np
import requests
import os


class StudentEnv(gym.Env):
    def __init__(self, traits=None):
        super(StudentEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(6)
        self.traits = traits
        self.timestep = 0
        self.max_days = 30
        self.days_until_deadline = 10
        self.days_deadline = 10
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)
        self.state = np.array([0.5, 0.5, 0.5, 0.5, self.days_until_deadline / self.max_days], dtype=np.float32)

    def step(self, action):
        energy, stress, self_efficacy, social_need, _ = self.state
        o = self.traits['openness']
        c = self.traits['conscientiousness']
        e = self.traits['extraversion']
        a = self.traits['agreeableness']
        n = self.traits['neuroticism']
        stress += 0.1 * (1 - self.days_until_deadline / self.days_deadline) * (1 + 0.5 * n)
        social_need = min(1, social_need + 0.05)
        self.days_until_deadline = max(0, self.days_until_deadline - 1)
        if self.days_until_deadline == 0:
            self.days_until_deadline = 10
            stress = min(1, stress + 0.2 * (1 + 0.5 * n))
        if action == 0:
            stress_threshold = 0.8 + 0.1 * c - 0.1 * n
            if stress > stress_threshold:
                energy -= 0.3 * (1 + 0.2 * n)
                stress += 0.05
                self_efficacy -= 0.05 * (1 + 0.3 * n)
            else:
                energy -= 0.2 * (1 - 0.2 * c)
                stress -= 0.15
                self_efficacy += 0.05
        elif action == 1:
            energy -= 0.15
            stress -= 0.05 * (1 + 0.2 * a)
            self_efficacy += 0.03 * (1 + 0.2 * a)
        elif action == 2:
            energy += 0.15 * (1 + 0.1 * o)
            stress -= 0.15
            social_need -= 0.1
            self_efficacy -= 0.01
        elif action == 3:
            energy -= 0.1 * (1 - 0.4 * e)
            social_need -= 0.3 * (1 + 0.3 * e)
            self_efficacy += 0.02
        elif action == 4:
            energy += 0.3
            stress -= 0.1 * (1 - 0.1 * (1 - c))
        elif action == 5:
            energy += 0.1
            stress += 0.05 * (1 + 0.2 * n)
            self_efficacy -= 0.05 * (1 + 0.3 * c)
        energy = np.clip(energy, 0, 1)
        stress = np.clip(stress, 0, 1)
        self_efficacy = np.clip(self_efficacy, 0, 1)
        social_need = np.clip(social_need, 0, 1)
        self.state = np.array([energy, stress, self_efficacy, social_need, self.days_until_deadline / self.max_days], dtype=np.float32)
        reward = self_efficacy - stress
        self.timestep += 1
        terminated = False
        truncated = self.timestep >= self.max_days
        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.traits = {
            'openness':          np.random.uniform(0, 1),
            'conscientiousness': np.random.uniform(0, 1),
            'extraversion':      np.random.uniform(0, 1),
            'agreeableness':     np.random.uniform(0, 1),
            'neuroticism':       np.random.uniform(0, 1)
        }
        self.days_until_deadline = 10
        self.state = np.array([0.5, 0.5, 0.5, 0.5, self.days_until_deadline / self.max_days], dtype=np.float32)
        self.timestep = 0
        return self.state, {}

## Basic interface

The loop below runs 30 steps with random actions. Replace `env.action_space.sample()` with your own logic.

In [ ]:
ACTION_NAMES = ["study", "attend_lecture", "leisure", "socialize", "rest", "skip"]

env = StudentEnv()
obs, _ = env.reset()

for step in range(30):
    action = env.action_space.sample()  # replace with your logic
    obs, reward, terminated, truncated, _ = env.step(action)
    energy, stress, self_efficacy, social_need, days_norm = obs
    print(f"Day {step+1:02d} | {ACTION_NAMES[action]:<16} | energy={energy:.2f}  stress={stress:.2f}  self_eff={self_efficacy:.2f}  social={social_need:.2f}  reward={reward:+.2f}")
    if terminated or truncated:
        break

## Generate diary entries

This cell runs a 30-day simulation and uses a local LLM (via Ollama) to write a diary entry for each day. Entries are saved to `synthetic_journals/`.

Requires Ollama running locally with the `gemma4:e4b` model. Adjust the model name below if needed.

In [ ]:
OLLAMA_MODEL = "gemma4:e4b"  # change if using a different model
os.makedirs("synthetic_journals", exist_ok=True)

env = StudentEnv()
obs, _ = env.reset()
prev_obs = None

for day in range(30):
    action = env.action_space.sample()  # replace with your logic
    obs, reward, done, truncated, _ = env.step(action)

    energy, stress, self_efficacy, social_need, days_norm = obs
    days_until_deadline = int(days_norm * env.max_days)
    action_name = ACTION_NAMES[action]

    prev_section = ""
    if prev_obs is not None:
        pe, ps, pse, psn, pd = prev_obs
        prev_section = f"Yesterday:\n  energy: {pe:.2f}, stress: {ps:.2f}, self_efficacy: {pse:.2f}, social_need: {psn:.2f}\n"

    prompt = f"""You are a student writing in your personal diary. Write a short, honest diary entry for today. Stay grounded — only describe things that match the event and state below. No invented events.

{prev_section}Today (day {env.timestep}):
  Action taken: {action_name}
  energy: {energy:.2f}
  stress: {stress:.2f}
  self_efficacy: {self_efficacy:.2f}
  social_need: {social_need:.2f}
  days until deadline: {days_until_deadline}
  reward signal: {reward:+.2f}

Write the diary entry in first person. 3-5 sentences. Reflect the emotional tone the state implies."""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0.0}}
    )
    diary_text = response.json()["response"].strip()

    filename = f"synthetic_journals/day_{env.timestep:02d}.txt"
    with open(filename, "w") as f:
        f.write(f"Day {env.timestep} | Action: {action_name}\n")
        f.write(f"State: energy={energy:.2f}, stress={stress:.2f}, self_efficacy={self_efficacy:.2f}, social_need={social_need:.2f}, deadline in {days_until_deadline}d\n")
        f.write("-" * 60 + "\n")
        f.write(diary_text + "\n")

    print(f"Day {env.timestep:02d} written — {action_name}")
    prev_obs = obs

    if done or truncated:
        break

print("Done — journals saved to synthetic_journals/")